<style>
body { font-family: Georgia, "Times New Roman", serif; line-height: 1.52; }
h1, h2, h3, h4 { font-family: Arial, Helvetica, sans-serif; }
h1 { margin-top: 0.9em; }
h2 { margin-top: 0.8em; }
.figure { text-align: center; margin: 1.15em auto; page-break-inside: avoid; }
.caption { font-size: 0.92em; color: #334155; text-align: left; max-width: 93%; margin: 0.45em auto 0 auto; }
table { font-size: 0.9em; border-collapse: collapse; width: 100%; margin: 0.7em 0; }
th, td { border: 1px solid #cbd5e1; padding: 6px; vertical-align: top; }
th { background: #f1f5f9; }
pre { white-space: pre-wrap; font-size: 0.84em; border: 1px solid #d0d7de; padding: 0.8em; border-radius: 6px; background: #f8fafc; }
code { font-size: 0.92em; }
.pagebreak { page-break-after: always; break-after: page; }
.small { font-size: 0.9em; color: #475569; }
.callout { border-left: 4px solid #2563eb; padding: 0.6em 0.8em; background: #eff6ff; margin: 0.8em 0; }
@media print {
  body { color: #111827; background: #ffffff; }
  h1, h2, h3, h4 { color: #0f172a; }
}
</style>

# English-to-Igbo Neural Machine Translation

## Comparing an RNN, a scratch Transformer, pretrained OPUS-MT, and fine-tuned OPUS-MT

**Course:** CS156, Finding Patterns in Data with Machine Learning  
**Term:** Spring 2026  
**Main deliverable:** this notebook exported as a PDF  
**Live translator demo:** [https://colab.research.google.com/drive/17k-6HDwkTttuWZnZR7vf9vdxOzKtmjEJ#scrollTo=VVcMBcSDxegp](https://colab.research.google.com/drive/17k-6HDwkTttuWZnZR7vf9vdxOzKtmjEJ#scrollTo=VVcMBcSDxegp)

**Live translator demo:**[https://github.com/Carldtitan/igbo_translator_modeling_CS156](https://github.com/Carldtitan/igbo_translator_modeling_CS156)

I began this final project with a different idea. After Assignment 2, I was considering my original project that classified events from my calendar. The feedback I received made me rethink that direction. A calendar classifier felt narrow, and I wanted the final project to be useful after the class ended. I pivoted to English-to-Igbo translation because I want to learn Igbo and because translation creates a real machine-learning pipeline: data cleaning, sequence modeling, training, inference, evaluation, and interpretation.

This report is written to stand on its own. The key code excerpts, row counts, model settings, final metrics, architecture diagrams, and the live Colab tester are included here so the project can be understood from the PDF alone.

<div class="pagebreak"></div>

# Table of Contents

1. Executive Summary  
2. What Is Included in This PDF  
3. Motivation and Research Question  
4. Data Sources  
5. Data Cleaning and Dataset Construction  
6. Exploratory Data Analysis  
7. Task Formulation  
8. Data Ethics and Scope  
9. Overall Pipeline  
10. Model 1, RNN LSTM Encoder-Decoder  
11. Model 2, Transformer From Scratch  
12. Model 3, Direct Pretrained OPUS-MT  
13. Model 4, Fine-Tuned OPUS-MT  
14. Prediction Generation and Metrics  
15. Metric Math  
16. Main Quantitative Results  
17. Hypothesis Revisited  
18. Results by Source  
19. Length and Exact Match Diagnostics  
20. Training Curves  
21. Qualitative Error Analysis  
22. Live Translator Demo  
23. Limitations  
24. Evidence Visible in This PDF  
25. Conclusion  
26. References  
27. Appendix: Visible Code Excerpts  

<div class="pagebreak"></div>

# 1. Executive Summary

I trained and evaluated four English-to-Igbo translation systems:

| Model | What it represents |
|---|---|
| RNN LSTM from scratch | A classical sequence-to-sequence baseline trained from scratch |
| Transformer from scratch | An attention-based encoder-decoder model trained only on my cleaned corpus |
| OPUS-MT pretrained | A pretrained English-to-Igbo translation model used directly |
| OPUS-MT fine-tuned | The pretrained OPUS-MT model adapted to my cleaned English-Igbo data |

The final comparison uses the same held-out test set for every model: 6,390 English-Igbo sentence pairs. The evaluation code generates one prediction set per model, normalizes text the same way for every system, and then computes BLEU, chrF, exact match, and prediction-to-reference length ratio. The important parts of that code are shown later in the report.

My original hypothesis was that direct pretrained OPUS-MT would clearly outperform my scratch Transformer. That was reasonable because OPUS-MT starts from external translation training, while my scratch Transformer only sees my cleaned corpus. The final result was more interesting. The scratch Transformer scored higher than direct pretrained OPUS-MT on BLEU, while direct pretrained OPUS-MT was nearly tied on chrF. After fine-tuning, OPUS-MT became the best model by a clear margin.

| Model | Rows | BLEU | chrF | Exact match | Length ratio |
| --- | ---: | ---: | ---: | ---: | ---: |
| RNN LSTM from scratch | 6390 | 7.46 | 20.68 | 0.0089 | 0.861 |
| Transformer from scratch | 6390 | 17.83 | 36.02 | 0.0105 | 1.0 |
| OPUS-MT pretrained | 6390 | 15.26 | 36.13 | 0.0028 | 1.201 |
| OPUS-MT fine-tuned | 6390 | 21.54 | 42.39 | 0.0083 | 1.03 |

The strongest model is **OPUS-MT fine-tuned**, with BLEU 21.54 and chrF 42.39. The strongest model trained fully from scratch is the **Transformer**, with BLEU 17.83 and chrF 36.02. The RNN was useful as a baseline because it showed the expected weaknesses of a recurrent sequence model on translation: short outputs, repeated phrases, and lower overlap with the references.

The refined conclusion is: **pretraining helped most when it was adapted to my own cleaned English-Igbo corpus through fine-tuning.**

<div class="pagebreak"></div>

# 2. What Is Included in This PDF

This report is meant to be read as the submission itself. I do not assume that the reader will inspect any outside code. The key pieces of the project are visible here:

| What is shown in this PDF | Why it matters |
|---|---|
| Data-source explanation and row counts | Shows what the translator learned from and what it was tested on |
| Cleaning code excerpts | Shows how raw Bible and JHW text became paired English-Igbo data |
| Model equations, architecture diagrams, and code excerpts | Explains how the RNN, Transformer, and OPUS-MT systems work |
| Training settings and visible code snippets | Shows the implementation choices behind each model |
| Final metric tables and figures | Shows the apples-to-apples comparison across all four models |
| Qualitative examples and error analysis | Shows what the translations look like beyond metric scores |
| Live Colab link | Lets a reader try new English sentences in a small UI |

The formal evaluation uses the full held-out test set of 6,390 rows. Every model is evaluated on the same examples, with the same normalization, and with the same metric code.

## 2.1 Live Translator Demo

The live tester is here:

[https://colab.research.google.com/drive/17k-6HDwkTttuWZnZR7vf9vdxOzKtmjEJ#scrollTo=VVcMBcSDxegp](https://colab.research.google.com/drive/17k-6HDwkTttuWZnZR7vf9vdxOzKtmjEJ#scrollTo=VVcMBcSDxegp)

To test the translator:

1. Open the Colab link.
2. Click **File -> Save a copy in Drive**.
3. Set **Runtime -> Change runtime type -> GPU**.
4. Run all cells.
5. Scroll to the translator UI at the bottom.
6. Type an English sentence and press **Translate**.
7. Compare the four project models with the Meta NLLB checker.

The fifth model in the live demo is `facebook/nllb-200-distilled-600M`. I use it as a strong external reference point, not as one of the four models in my formal apples-to-apples evaluation.

<div class="pagebreak"></div>

# 3. Motivation and Research Question

This project came from a pivot. My earlier idea was to continue from Assignment 2 and classify events from my calendar. After the feedback on that assignment, I realized that the final project needed a clearer purpose and a stronger technical reason to exist. English-to-Igbo translation fit both goals.

I chose Igbo because I want to learn the language. I want to understand full sentences instead of isolated vocabulary lists. A translator lets me type an English sentence, compare several Igbo outputs, and notice patterns in word order, function words, and phrasing. That makes the project useful to me outside the assignment.

The project also matters beyond my own learning. Igbo is spoken by many people, yet it has less public language technology than English and other high-resource languages. This is the low-resource language problem: a language can have many speakers and still have fewer aligned datasets, fewer pretrained systems, and fewer evaluation benchmarks.

My research question is:

**How much does transfer learning help English-to-Igbo translation when compared with models trained from scratch on the same cleaned data?**

This question has a practical answer. If I want to build a useful English-to-Igbo translator, I need to know whether the best path is training from scratch, using a pretrained model directly, or adapting a pretrained model to my data.

My hypothesis before the final evaluation was:

$$
\text{OPUS-MT pretrained} \; > \; \text{scratch Transformer} \; > \; \text{RNN}
$$

I expected this because OPUS-MT already had translation knowledge from outside my project. The final BLEU result changed that hypothesis. The scratch Transformer beat direct OPUS-MT on BLEU, then fine-tuned OPUS-MT beat every model.

<div class="pagebreak"></div>

# 4. Data Sources

The project uses English-Igbo parallel text. Each row has an English input and an Igbo target. This is supervised learning because the model sees the desired output during training.

I used two sources:

| Source | Why it is useful | Limitation |
|---|---|---|
| English and Igbo Bible verse data | Verse IDs make sentence-pair alignment possible | Religious and formal style |
| JHW English-Igbo parallel text | Much larger source of aligned examples | Style is still not everyday conversation |

The final cleaned data used in the report has this composition:

| Split | Source | Rows |
| --- | --- | ---: |
| train | Bible | 27986 |
| train | JHW | 506078 |
| train | Combined total | 534064 |
| test | Bible | 3109 |
| test | JHW | 3281 |
| test | Combined total | 6390 |

The most important data fact is that the training data is dominated by JHW rows, while the final test set is nearly balanced between Bible and JHW rows. That affects interpretation because a model can learn mostly JHW-style training text and then face a large Bible portion during evaluation.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/01_dataset_composition.png" style="max-width:96%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 1. Dataset composition by source for the cleaned train and test splits.</strong></p>
</div>

<div class="pagebreak"></div>

# 5. Data Cleaning and Dataset Construction

The cleaning stage had to turn two different sources into one consistent translation dataset. Every final row has exactly two text fields, one English sentence or verse and one Igbo sentence or verse. This common schema matters because the four models use different libraries, but they all need the same supervised pair structure:

$$
(\text{English input}, \text{Igbo target})
$$

The Bible source required the most custom processing. The raw Bible files were VPL SQL exports, so the data was stored inside SQL `INSERT` lines rather than in a ready-made CSV or JSONL table. I parsed only verse rows, normalized Unicode with NFC, standardized curly quotes, collapsed repeated whitespace, and kept the verse identifier. The key decision was to align English and Igbo by `verseID`. That is safer than trying to align by row number, because row order can shift if one file contains a missing or extra verse.

The JHW source was already closer to model-ready data. It already had English and Igbo fields and already came with train and test splits. I still checked the schema, inspected samples, measured missing values, and removed rows where either side was blank. After that, I kept only the English and Igbo fields so the JHW rows matched the Bible rows.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/09_data_cleaning_lineage.png" style="max-width:98%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 2. Data lineage from the raw Bible and JHW sources into the final combined train and test files.</strong></p>
</div>

## 6.1 Bible Cleaning Logic

The Bible data already contains verse identifiers. I used those identifiers to align English verses with Igbo verses. An inner join keeps only verses that appear in both languages. I kept verses intact rather than splitting them into smaller sentences, because verse-level alignment was the reliable unit provided by the source files.

```python
VERSE_PATTERN = re.compile(
    r'INSERT INTO \w+ VALUES $"([^"]*)","([^"]*)","([^"]*)","([^"]*)","([^"]*)","([^"]*)","(.*)"$;$'
)

def normalize_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u02bc", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()
```

```python
english_df = load_english_verses().rename(columns={"verseText": "English"})
igbo_df = load_igbo_verses().rename(columns={"verseText": "Igbo"})

aligned_df = english_df.merge(
    igbo_df[["verseID", "Igbo"]],
    on="verseID",
    how="inner",
)

clean_bible_df = aligned_df[
    aligned_df["English"].str.strip().ne("") &
    aligned_df["Igbo"].str.strip().ne("")
].copy()
```

After alignment, I kept only the translation columns and made a reproducible 90/10 split with random seed 42. That created 27,986 Bible train rows and 3,109 Bible test rows.

## 6.2 JHW Cleaning Logic

The JHW data already came as train and test splits. I filtered out rows where either side was missing or blank. I kept the original JHW split because it was the split provided with the corpus.

```python
clean_train = train_split.filter(
    lambda row: bool(
        row["English"] and row["English"].strip()
        and row["Igbo"] and row["Igbo"].strip()
    )
)

clean_test = test_split.filter(
    lambda row: bool(
        row["English"] and row["English"].strip()
        and row["Igbo"] and row["Igbo"].strip()
    )
)
```

## 6.3 Combining the Sources

Once the Bible and JHW data had the same two-column structure, I concatenated the training splits and test splits.

```python
combined_train = pd.concat([bible_train, jhw_train], ignore_index=True)
combined_test = pd.concat([bible_test, jhw_test], ignore_index=True)
```

The final cleaned files contain no blank English rows and no blank Igbo rows. The combined train set has 534,064 rows. The combined test set has 6,390 rows. The test set is balanced enough to compare source behavior: 3,109 Bible rows and 3,281 JHW rows.

## 6.4 Duplicate and Split Check

A stricter machine translation benchmark would remove exact duplicate sentence pairs before splitting. I checked this because repeated religious or news-style phrases can appear many times. The Bible split had 35 exact test pairs that also appeared in Bible train, which is 1.13 percent of the Bible test set. The JHW split had 2,735 exact test pairs that also appeared in JHW train, which is 83.36 percent of the JHW test set.

This means the final comparison is fair across the four project models because each model receives the same test rows, but the absolute scores should be read as performance on this cleaned project benchmark rather than as a complete claim about general English-to-Igbo generalization. The by-source results are important for that reason. They show how performance differs between the Bible rows and the JHW rows instead of hiding both sources inside one number.

<div class="pagebreak"></div>


# 6. Exploratory Data Analysis

The EDA has two purposes. First, it checks whether the train and test sets have the expected source composition. Second, it checks how long the sentences are, because sequence length affects translation difficulty.

The held-out test set contains both short and long sentences. Longer sentences are harder because the model must preserve more content and handle more alignment decisions. The RNN is especially sensitive to this because its decoder depends on a recurrent hidden state. The Transformer and OPUS-MT have attention mechanisms that make long-range alignment easier.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/02_test_sentence_lengths.png" style="max-width:96%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 3. Held-out sentence length distributions after the shared metric tokenization.</strong></p>
</div>

The final evaluation script computes token counts using the same metric normalization used for BLEU and chrF. That prevents the report from using one tokenization for EDA and another tokenization for evaluation.

```python
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def normalize_for_metric(text: str) -> str:
    text = str(text).lower().strip()
    text = text.replace("\u2019", "'")
    text = text.replace("\u2018", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = normalize_whitespace(text)
    return " ".join(TOKEN_PATTERN.findall(text))
```

<div class="pagebreak"></div>

# 7. Task Formulation

The machine learning task is supervised sequence-to-sequence translation. The dataset is:

$$
\mathcal{D} = \{(x^{(i)}, y^{(i)})\}_{i=1}^N
$$

where $x$ is an English token sequence and $y$ is an Igbo token sequence.

For one example:

$$
x = (x_1, x_2, \ldots, x_m)
$$

$$
y = (y_1, y_2, \ldots, y_n)
$$

A neural translation model estimates:

$$
p_\theta(y \mid x)
$$

The decoder generates Igbo tokens from left to right:

$$
p_\theta(y \mid x) = \prod_{t=1}^n p_\theta(y_t \mid y_{<t}, x)
$$

Training minimizes the negative log likelihood of the target sequence:

$$
\mathcal{L}(\theta; x,y) = -\sum_{t=1}^n \log p_\theta(y_t \mid y_{<t}, x)
$$

This is why the code uses cross-entropy losses. At each decoder step, the model predicts a probability distribution over the next Igbo token.

In implementation terms, each target sentence is shifted by one position during training. The decoder receives the beginning-of-sentence token and the previous true target tokens as input. It is trained to predict the next target token. For a target sequence with special tokens, the training pair looks like this:

$$
\text{decoder input} = (\langle sos \rangle, y_1, y_2, \ldots, y_{n-1})
$$

$$
\text{decoder target} = (y_1, y_2, \ldots, y_n, \langle eos \rangle)
$$

Padding tokens are ignored in the loss. That matters because English and Igbo sentences have different lengths, and mini-batches require padding to make rectangular tensors.

<div class="pagebreak"></div>


# 8. Data Ethics and Scope

The data decision in this project was about copyright, dialect, and dataset size. I wanted the project to connect to my goal of learning Igbo, but I also needed enough aligned English-Igbo examples to train and compare four models.

For the Bible source, I used the World English Bible (WEB) on the English side. The WEB version is public domain and not copyrighted, according to eBible's copyright statement. That made it a better fit for this project than using a copyrighted English Bible translation. The verse structure also made the Bible data useful because English and Igbo verses could be aligned with verse identifiers.

I also considered a more personal data source. I do not know how to speak Igbo yet, so my original plan was to ask my dad to help create or check sentences. That would have connected directly to my reason for building the translator. The problem was dialect. Igbo has many dialects, and the dialect my father speaks is quite different from the more contemporary or general written Igbo I wanted the models to learn from.

That left me with two choices:

| Option | Strength | Weakness |
|---|---|---|
| Ask my dad to create a smaller sentence set | More personal and closer to my family background | Too small for training four neural translation systems, and dialect-specific |
| Use larger public English-Igbo data | Enough examples to train and evaluate all four models | Less personal, and the style is shaped by Bible and JHW text |

I chose the public-data route because this was a machine learning project, and the models needed enough aligned examples to learn useful translation patterns. It would still have been valuable to get my dad's help for qualitative checking, especially to understand whether a translation sounds natural in his dialect. I did not use family-created sentences as training data for this version because the public data gave me more coverage and a clearer evaluation setup.

The scope of the project is therefore:

| Question | Answer for this project |
|---|---|
| Why use the WEB Bible text? | It is public domain and not copyrighted, so it was appropriate for a student ML project. |
| Why not build the training data from my dad's dialect? | The dataset would have been small and dialect-specific. |
| Does the model learn conversational Igbo? | Not fully. Bible and JHW text are aligned and useful, but their style is not everyday conversation. |
| Is the evaluation fair across my four models? | Yes. All four models use the same held-out `Combined_test.jsonl` rows. |
| Is the evaluation a full human translation judgment? | No. BLEU and chrF are automatic overlap metrics, so human judgment would still be needed. |

This is why the report separates two claims. The technical claim is that fine-tuned OPUS-MT is best under my automatic test-set evaluation. The practical claim is narrower: the model is a useful starting point for my Igbo learning, especially when its outputs are checked against another model or a fluent speaker.

<div class="pagebreak"></div>


# 9. Overall Pipeline

The project has three big stages: build the dataset, train or load translation models, then evaluate every model on the same held-out rows. The important design rule is that the model training paths are allowed to differ, but the final evaluation path is shared.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/10_evaluation_harness.png" style="max-width:98%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 4. Final evaluation harness. Each model produces 6,390 predictions for the same held-out rows before shared normalization and metrics.</strong></p>
</div>

The final evaluation design matters because earlier development runs used different limits. For example, one Transformer development run translated only 1,000 test examples. The final comparison fixes that by using the full 6,390-row held-out test set for every model.

This is what makes the comparison apples-to-apples: same test rows, same reference translations, same metric code, and the same source labels for Bible versus JHW analysis.

<div class="pagebreak"></div>


# 10. Model 1, RNN LSTM Encoder-Decoder

The RNN model is the classical neural baseline. It is a sequence-to-sequence model with an LSTM encoder and LSTM decoder. The encoder reads the English sentence and compresses it into hidden and cell states. The decoder uses those states to generate Igbo tokens.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/11_rnn_tensor_architecture.png" style="max-width:98%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 5. RNN LSTM encoder-decoder architecture with tensor shapes, teacher forcing, and loss path.</strong></p>
</div>

The source and target sentences are regex-tokenized and capped at 40 tokens. English tokens are mapped through a training-set vocabulary and embedded into 256-dimensional vectors. The encoder is a two-layer bidirectional LSTM with hidden size 384. Because the encoder is bidirectional, its output width is 768. Linear bridge layers convert the bidirectional hidden and cell states into the two-layer decoder state size. The decoder is a two-layer LSTM with hidden size 384, followed by a linear output layer over the Igbo vocabulary.

During training, the RNN uses teacher forcing. With probability 0.6, the decoder receives the true previous Igbo token. With probability 0.4, it receives its own previous prediction. That balances stable training with exposure to the kind of mistakes the decoder will face at inference time.

$$
\tilde{y}_{t-1} =
\begin{cases}
y_{t-1} & \text{with probability } 0.6 \\
\hat{y}_{t-1} & \text{with probability } 0.4
\end{cases}
$$

The key LSTM equations are:

$$
i_t = \sigma(W_i x_t + U_i h_{t-1} + b_i)
$$

$$
f_t = \sigma(W_f x_t + U_f h_{t-1} + b_f)
$$

$$
o_t = \sigma(W_o x_t + U_o h_{t-1} + b_o)
$$

$$
\tilde{c}_t = \tanh(W_c x_t + U_c h_{t-1} + b_c)
$$

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

$$
h_t = o_t \odot \tanh(c_t)
$$

The decoder turns its hidden state into a vocabulary distribution with a linear layer and softmax:

$$
p_\theta(y_t \mid y_{<t}, x) = \text{softmax}(W_{out}h_t + b_{out})
$$

The actual training configuration used all available training rows, batch size 64, 10 configured epochs, 8 completed epochs in the saved history, dropout 0.3, learning rate 0.0005, and teacher forcing ratio 0.6.

```python
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0,
                            bidirectional=True, batch_first=True)
        self.hidden_bridge = nn.Linear(hidden_dim * 2, hidden_dim)
        self.cell_bridge = nn.Linear(hidden_dim * 2, hidden_dim)
```

The RNN training loss decreased from 4.81 to 3.24 across the saved history, and validation loss decreased from 5.31 to 4.90. The model learned signal, yet the final test BLEU remained low. Translation creates a bottleneck for this model because the decoder depends heavily on the compressed encoder state. The bidirectional encoder helps, but the model has no explicit cross-attention mechanism that lets every generated Igbo token look back at every English token.

<div class="pagebreak"></div>


# 11. Model 2, Transformer From Scratch

The scratch Transformer is the strongest model in the project that was trained from zero. It uses attention instead of recurrence. This matters because translation is partly an alignment problem. The model needs to learn which English words or phrases are relevant when producing each Igbo token.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/12_transformer_tensor_architecture.png" style="max-width:98%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 6. Scratch Transformer architecture with encoder states, decoder states, cross-attention, logits, and masked loss.</strong></p>
</div>

The scratch Transformer uses regex tokenization with 40-token caps for source and target sequences. It uses learned token embeddings and learned position embeddings with dimension 256. The encoder has two blocks. Each block uses four-head self-attention followed by a feed-forward layer with hidden dimension 512. The decoder also has two blocks. Each decoder block uses masked self-attention, cross-attention to the encoder output, and a feed-forward layer.

The central attention equation is:

$$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

For the encoder, $Q$, $K$, and $V$ all come from the English sequence. That lets each English token become contextualized by the other English tokens. For decoder masked self-attention, $Q$, $K$, and $V$ come from the partially generated Igbo sequence, and a causal mask blocks future target positions. For cross-attention, the queries come from the Igbo decoder states, while keys and values come from the English encoder states.

Multi-head attention repeats this calculation in several representation spaces:

$$
\text{head}_j = \text{Attention}(QW_j^Q, KW_j^K, VW_j^V)
$$

$$
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H)W^O
$$

The feed-forward sublayer applies the same two-layer nonlinear transformation to every position:

$$
\text{FFN}(z) = W_2\,\text{ReLU}(W_1z + b_1) + b_2
$$

The implementation also uses residual connections and layer normalization around the attention and feed-forward sublayers. The training setup used all rows, 8 epochs, batch size 64, embedding dimension 256, feed-forward dimension 512, 4 attention heads, 2 encoder blocks, 2 decoder blocks, dropout 0.1, and learning rate 0.0003.

The masked loss ignores padding tokens:

```python
@keras.utils.register_keras_serializable(package="cs156")
def masked_loss(y_true, y_pred):
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    mask = tf.cast(tf.not_equal(y_true, 0), loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.maximum(tf.reduce_sum(mask), 1.0)
```

The Transformer training history shows steady improvement: training loss moved from 3.11 to 1.79 and validation loss moved from 2.36 to 1.78 over 8 epochs. This supports the final result that the scratch Transformer learned a meaningful translation distribution from the cleaned corpus.

The scratch Transformer had no external pretraining, yet it trained directly on the same cleaned corpus family used in final evaluation. Under the fixed test distribution in this project, that corpus fit helped it beat direct OPUS-MT on BLEU.

<div class="pagebreak"></div>


# 12. Model 3, Direct Pretrained OPUS-MT

OPUS-MT is an open neural machine translation project based on Marian NMT. I used the Hugging Face model `Helsinki-NLP/opus-mt-en-ig`, which is an English-to-Igbo translation model.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/13_opus_transfer_architecture.png" style="max-width:98%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 7. OPUS-MT transfer-learning design. Direct OPUS-MT keeps pretrained weights fixed, while fine-tuned OPUS-MT adapts the same model to the project corpus.</strong></p>
</div>

The saved OPUS-MT configuration shows a MarianMT encoder-decoder model with 6 encoder layers, 6 decoder layers, 8 attention heads on each side, model dimension 512, feed-forward dimension 2048, vocabulary size 56,127, maximum position embeddings 512, swish activation, and dropout 0.1.

OPUS-MT uses subword tokenization. This is important for Igbo because a word-level vocabulary can become sparse quickly. Subword pieces let the model represent unseen or rare words as smaller units. The model is still encoder-decoder translation, but it begins from parameters trained before this project.

The direct pretrained model tests a simple transfer-learning baseline:

$$
\hat{y} = \arg\max_y p_{\theta_0}(y \mid x)
$$

where $\theta_0$ is the pretrained OPUS-MT parameter set. In this model, I keep the weights fixed on my cleaned corpus. I only load the pretrained model and generate translations for the test set. This makes it a transfer baseline: it measures how well an existing English-to-Igbo model matches my Bible and JHW benchmark without adaptation.

```python
MODEL_NAME = "Helsinki-NLP/opus-mt-en-ig"
MODEL_LABEL = "direct_pretrained"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
```

I expected this model to be very strong because it starts from external English-to-Igbo translation training. The final BLEU score was 15.26, which was lower than the scratch Transformer BLEU of 17.83. The chrF score was 36.13, almost equal to the scratch Transformer chrF of 36.02.

This means direct OPUS-MT was often close at the character-fragment level, while the scratch Transformer matched the exact token n-grams of my test set better.

<div class="pagebreak"></div>


# 13. Model 4, Fine-Tuned OPUS-MT

The fine-tuned OPUS-MT model starts with the same pretrained `Helsinki-NLP/opus-mt-en-ig` checkpoint and continues training on every row in the combined training set. Figure 7 shows the difference between the direct and fine-tuned OPUS-MT paths.

In transfer-learning notation, fine-tuning starts from $\theta_0$ and learns adapted parameters $\theta^*$:

$$
\theta^* = \arg\min_{\theta} \frac{1}{N}\sum_{i=1}^{N} -\log p_{\theta}\left(y^{(i)} \mid x^{(i)}\right)
$$

with initialization:

$$
\theta \leftarrow \theta_0
$$

This equation means the model begins with translation knowledge from OPUS-MT, then updates the same parameters to reduce negative log likelihood on my 534,064 English-Igbo training pairs. The target labels are token IDs produced by the OPUS tokenizer. Padding labels are ignored by the sequence-to-sequence trainer, so the loss focuses on real target tokens.

The fine-tuning run used one epoch over all 534,064 training rows, 2,000 monitor evaluation rows, max source and target length 96, per-device train batch size 32, gradient accumulation 1, and one-beam generation for saved predictions. The settings were designed to make Colab training faster while still using the full dataset.

```python
FINE_TUNE_TRAIN_SAMPLE_SIZE = None
FINE_TUNE_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 32
AUTO_FIND_BATCH_SIZE = True
USE_ADAFACTOR = True
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 96
GROUP_BY_LENGTH = True
```

Length grouping helped because translation batches become inefficient when very short and very long examples are mixed together. Truncating at 96 source and target tokens kept unusually long examples from dominating GPU memory. Adafactor helped because it is commonly used for large Transformer fine-tuning with lower optimizer memory than Adam-style second-moment tensors. I used one epoch because one full pass over 534,064 rows was already a large amount of update signal for a pretrained translation model.

The fine-tuned model had the best final performance: BLEU 21.54 and chrF 42.39. Compared with direct OPUS-MT, fine-tuning gained 6.28 BLEU and 6.26 chrF. Compared with the scratch Transformer, it gained 3.71 BLEU and 6.37 chrF. Compared with the RNN, it gained 14.08 BLEU and 21.71 chrF.

Fine-tuning was the best technical choice because it matched the structure of the problem. English-to-Igbo translation is a specialized task with limited public data. Starting from OPUS-MT gives the model a translation prior, then fine-tuning adapts that prior to my corpus.

<div class="pagebreak"></div>


# 14. Prediction Generation and Metrics

The final evaluation loads the fixed test set, generates or aligns predictions for all four models, and computes metrics with one shared normalization function. The evaluation flow is shown in Figure 4.

The evaluation starts from `Combined_test.jsonl`. The file is loaded into a dataframe, the Igbo column is renamed to `Reference Igbo`, and a `row_id` column is added. That row id is important because prediction files from different models must line up with the same English input and the same reference translation.

```python
test_df = pd.DataFrame(read_jsonl(data_dir / "Combined_test.jsonl"))
test_df = test_df.rename(columns={"Igbo": "Reference Igbo"})
test_df.insert(0, "row_id", np.arange(len(test_df), dtype=int))
```

The shared metric normalization lowercases text, standardizes curly quotes, collapses whitespace, and applies one regex tokenizer. I used the same function for every prediction and every reference.

```python
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

def normalize_for_metric(text: str) -> str:
    text = str(text).lower().strip()
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u02bc", "'")
    text = normalize_whitespace(text)
    return " ".join(TOKEN_PATTERN.findall(text))
```

Each model produces one prediction row per test row. The RNN predictions come from a saved PyTorch checkpoint with greedy decoding. The Transformer predictions come from a saved Keras model with autoregressive decoding. The OPUS-MT predictions come from Hugging Face MarianMT models or full-test saved preview files. In all four cases, the final prediction file contains 6,390 rows.

```python
def compute_metric_row(model_slug: str, pred_df: pd.DataFrame, split_label: str) -> dict:
    refs = pred_df["Reference Igbo"].map(normalize_for_metric).tolist()
    preds = pred_df["Predicted Igbo"].fillna("").map(normalize_for_metric).tolist()

    exact_match = float(np.mean([pred == ref for pred, ref in zip(preds, refs)]))
    bleu = BLEU(tokenize="none", effective_order=True).corpus_score(preds, [refs]).score
    chrf = CHRF().corpus_score(preds, [refs]).score

    ref_lengths = np.array([max(token_count(text), 1) for text in pred_df["Reference Igbo"]], dtype=float)
    pred_lengths = np.array([token_count(text) for text in pred_df["Predicted Igbo"].fillna("")], dtype=float)
    length_ratios = pred_lengths / ref_lengths
```

The call to SacreBLEU uses `tokenize="none"` because I already normalize and tokenize the strings before sending them to the metric. The reference argument has shape `[refs]` because SacreBLEU expects a list of reference streams, even when there is only one reference translation per sentence.

BLEU and chrF are the main automatic translation-quality metrics. Exact match is included as a strict sanity check. Length ratio is a diagnostic for under-translation and over-translation. By-source BLEU and chrF show whether performance changes between Bible rows and JHW rows.

<div class="pagebreak"></div>


# 15. Metric Math

BLEU and chrF are overlap metrics. They do not understand meaning directly, yet they give a reproducible way to compare translation systems on the same test set. This project uses them as automatic evidence, then interprets them alongside length diagnostics and qualitative examples.

## 16.1 BLEU

BLEU uses modified n-gram precision and a brevity penalty. For n-gram order $n$, modified precision is:

$$
p_n = \frac{\sum_{g \in G_n(\hat{y})} \min\left(\text{count}_{\hat{y}}(g),\text{count}_{y}(g)\right)}{\sum_{g \in G_n(\hat{y})} \text{count}_{\hat{y}}(g)}
$$

Here, $\hat{y}$ is the model prediction, $y$ is the reference, and $G_n(\hat{y})$ is the set of n-grams in the prediction. The clipping step prevents a model from gaining unlimited credit by repeating the same phrase.

The corpus BLEU score is:

$$
BLEU = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)
$$

The brevity penalty is:

$$
BP =
\begin{cases}
1, & c > r \\
\exp(1-r/c), & c \le r
\end{cases}
$$

where $c$ is total candidate length and $r$ is total reference length. Short translations are punished by the brevity penalty. Long translations can also lose precision because they introduce extra n-grams that are absent from the reference.

## 16.2 chrF

chrF uses character n-gram precision and recall. It is useful in translation because a word can be partly correct even when exact word-level matching fails. The simplified F-score is:

$$
chrF_{\beta} = (1 + \beta^2) \cdot \frac{chrP \cdot chrR}{\beta^2 \cdot chrP + chrR}
$$

where $chrP$ is character n-gram precision and $chrR$ is character n-gram recall. In this project, chrF is especially useful because Igbo spelling, punctuation, and morphology can make exact word overlap too harsh.

## 16.3 Exact Match

Exact match is:

$$
\text{Exact Match} = \frac{1}{M}\sum_{i=1}^{M}\mathbf{1}\left[\text{normalize}(\hat{y}^{(i)}) = \text{normalize}(y^{(i)})\right]
$$

This metric is intentionally strict. A good translation can use different wording from the reference, so exact match should be low for a real translation task. I include it as a sanity check because it is easy to interpret.

## 16.4 Length Ratio

The length ratio for one row is:

$$
\text{Length Ratio}^{(i)} = \frac{\operatorname{tokens}(\hat{y}^{(i)})}{\max\left(\operatorname{tokens}(y^{(i)}),1\right)}
$$

The report averages this ratio across the test set. Values below 1 suggest under-translation. Values above 1 suggest longer outputs than the reference. A value close to 1 does not prove correctness, yet it shows better length calibration.

<div class="pagebreak"></div>


# 16. Main Quantitative Results

The main result table is:

| Model | Rows | Exact match | BLEU | chrF | Avg predicted tokens | Avg reference tokens | Length ratio |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| RNN LSTM from scratch | 6390 | 0.0089 | 7.46 | 20.68 | 24.63 | 31.59 | 0.861 |
| Transformer from scratch | 6390 | 0.0105 | 17.83 | 36.02 | 27.41 | 31.59 | 1.0 |
| OPUS-MT pretrained | 6390 | 0.0028 | 15.26 | 36.13 | 33.02 | 31.59 | 1.201 |
| OPUS-MT fine-tuned | 6390 | 0.0083 | 21.54 | 42.39 | 30.33 | 31.59 | 1.03 |

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/04_bleu_chrf_comparison.png" style="max-width:96%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 8. BLEU and chrF scores for all four project models on the same test set.</strong></p>
</div>

The ranking by BLEU is:

$$
\text{OPUS-MT fine-tuned} > \text{Transformer scratch} > \text{OPUS-MT pretrained} > \text{RNN LSTM}
$$

The ranking by chrF is:

$$
\text{OPUS-MT fine-tuned} > \text{OPUS-MT pretrained} \approx \text{Transformer scratch} > \text{RNN LSTM}
$$

The difference between BLEU and chrF is important. BLEU rewards exact token n-gram overlap. chrF rewards character n-gram overlap. Direct OPUS-MT and the scratch Transformer were almost tied on chrF, which suggests direct OPUS-MT often produced related word forms or character fragments. The Transformer had stronger exact token n-gram overlap with the reference set.

<div class="pagebreak"></div>

# 17. Hypothesis Revisited

My original hypothesis was that direct pretrained OPUS-MT would beat the scratch Transformer by a large amount. I expected that because OPUS-MT already had English-to-Igbo translation knowledge, while the scratch Transformer learned from my project data only.

The final result was:

| Comparison | BLEU | chrF |
|---|---:|---:|
| Scratch Transformer | 17.83 | 36.02 |
| Direct pretrained OPUS-MT | 15.26 | 36.13 |
| Fine-tuned OPUS-MT | 21.54 | 42.39 |

The reason the hypothesis was proven wrong is that pretraining and distribution fit are different. Direct OPUS-MT may know general translation patterns, yet the scratch Transformer was trained on the same cleaned distribution used in the final test set. BLEU rewards exact phrase overlap with that reference distribution. This gives the scratch Transformer an advantage on BLEU.

Fine-tuning combined both strengths. OPUS-MT started from a pretrained translation model, then adapted to the exact corpus. That is why the fine-tuned OPUS-MT model became strongest.

This is the main learning point for me: a pretrained model is not automatically best in the exact evaluation setting. The match between training data, evaluation data, tokenization, and generation settings matters.

## 18.1 Why the Unexpected Result Makes Sense

The unexpected result was:

$$
BLEU(\text{scratch Transformer}) = 17.83 > BLEU(\text{direct OPUS-MT}) = 15.26
$$

This makes sense for four reasons.

First, the scratch Transformer trained on `Combined_train.jsonl`, and the final test set came from the same cleaned data pipeline. BLEU rewards exact n-gram overlap, so distribution fit matters.

Second, direct OPUS-MT used its own pretrained tokenization and translation habits. Those habits may be good in general while still producing sentences that differ from the reference phrasing in this project.

Third, the direct OPUS-MT length ratio was 1.201. Longer outputs can reduce BLEU precision. The scratch Transformer length ratio was 1.000, which is almost perfectly calibrated to reference length on average.

Fourth, chrF tells a softer story. Direct OPUS-MT scored 36.13 chrF, and the scratch Transformer scored 36.02. This suggests direct OPUS-MT was not failing completely. It was often close at the character level, even when its word n-gram overlap was lower.

Fine-tuning resolves the tension. Once OPUS-MT trained on `Combined_train.jsonl`, it gained the corpus fit that direct OPUS-MT lacked. That is why its BLEU rose to 21.54 and chrF rose to 42.39.

<div class="pagebreak"></div>

# 18. Results by Source

The final evaluation labels each test row as Bible or JHW by matching it back to the cleaned source-specific test sets. This matters because the sources have different styles.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/06_metrics_by_source.png" style="max-width:96%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 9. BLEU and chrF separated by Bible and JHW test rows.</strong></p>
</div>

Every model scored better on Bible rows than on JHW rows by BLEU. The RNN scored BLEU 9.15 on Bible rows and 5.32 on JHW rows. The scratch Transformer scored 19.45 on Bible and 15.77 on JHW. Direct OPUS-MT scored 17.85 on Bible and 11.40 on JHW. Fine-tuned OPUS-MT scored 24.38 on Bible and 17.89 on JHW.

The chrF pattern is similar. Fine-tuned OPUS-MT was strongest on both sources: chrF 43.79 on Bible and 40.96 on JHW. This source-specific result supports the overall conclusion. The best model was not winning only because of one source.

The source split also matters because the JHW train/test data contained many exact repeated pairs across splits. Reporting Bible and JHW separately makes that limitation visible instead of hiding both sources inside one overall score.

<div class="pagebreak"></div>


# 19. Length and Exact Match Diagnostics

Length ratio is a useful diagnostic because translation models can fail by saying too little or by adding extra content.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/07_prediction_length_ratio.png" style="max-width:96%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 10. Prediction-to-reference length ratio for each model.</strong></p>
</div>

The RNN had a length ratio of 0.861. Its outputs were shorter than the references on average. This matches under-translation and early stopping behavior.

The direct pretrained OPUS-MT model had a length ratio of 1.201. Its outputs were longer than the references on average. This helps explain why chrF was strong while BLEU was lower. Extra text can preserve many character fragments and still reduce exact n-gram precision.

The fine-tuned OPUS-MT model had a length ratio of 1.030. This is close to the reference length, which supports the idea that fine-tuning calibrated the model to my corpus.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/05_exact_match_comparison.png" style="max-width:96%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 11. Exact match rates. This metric is intentionally strict for translation.</strong></p>
</div>

Exact match is low for all models because translation allows more than one correct sentence. This result is expected and should not be treated as the main success criterion.

<div class="pagebreak"></div>

# 20. Training Curves

The training curves show that the scratch models learned from the data.

<div class="figure">
  <img src="Final%20Evaluation%20Outputs/figures/03_training_curves.png" style="max-width:96%; height:auto; border: 1px solid #d0d7de;" />
  <p class="caption"><strong>Figure 12. Training loss curves for the RNN, scratch Transformer, and fine-tuned OPUS-MT.</strong></p>
</div>

The RNN training loss decreased across saved epochs, yet its validation loss stayed high. That is consistent with its weaker final translation performance.

The Transformer showed smoother improvement in both training and validation loss. That matches its stronger final BLEU.

The OPUS fine-tuning run completed one full epoch over 534,064 rows. The saved training result reported train loss 1.179 and runtime 1,104 seconds in Colab. Since it started from pretrained weights, one epoch was enough to improve substantially on the final test set.

The main comparison is still the final held-out test evaluation. Training curves show learning behavior, while test metrics show out-of-sample performance.

<div class="pagebreak"></div>

# 21. Qualitative Error Analysis

The qualitative examples help explain what the metrics mean.

The main error patterns are:

| Error type | Meaning | Model where it is clearest |
|---|---|---|
| Under-translation | The output leaves out content | RNN LSTM |
| Repetition | The decoder repeats words or phrases | RNN LSTM |
| Over-translation | The output is too long | Direct OPUS-MT |
| Lexical drift | The output uses related wording with shifted meaning | All models |
| Domain-style mismatch | The output sounds more like one source than the reference | Scratch and pretrained models |

For my personal goal, the qualitative results matter a lot. I am trying to build a model that helps me study Igbo sentences. The fine-tuned OPUS-MT outputs are the most useful starting point because they are closer to the references and less likely to collapse into repeated phrases.

The Meta NLLB checker in the live demo gives another external output to compare against. I do not treat it as ground truth. It is useful because it gives a strong multilingual model's translation for the same English sentence.

## 22.1 What This Means for My Igbo Learning

For learning Igbo, the best output is not always the output with the highest automatic score. I need translations that help me notice structure. The live tester is useful because I can type a sentence and compare five outputs: my four project models and Meta NLLB. When the fine-tuned OPUS-MT and Meta NLLB agree, I have more confidence. When they differ, I know that sentence needs human checking.

This makes the project useful after the class ends. I can use it as a phrase exploration tool, not as a final authority. The model can show me patterns, and a fluent speaker can help me confirm which patterns are natural.

<div class="pagebreak"></div>

# 22. Live Translator Demo

The live tester makes the project easier to inspect. It is separate from the fixed test-set evaluation. It lets a reader type new English sentences and compare outputs.

Colab link:

[https://colab.research.google.com/drive/17k-6HDwkTttuWZnZR7vf9vdxOzKtmjEJ#scrollTo=VVcMBcSDxegp](https://colab.research.google.com/drive/17k-6HDwkTttuWZnZR7vf9vdxOzKtmjEJ#scrollTo=VVcMBcSDxegp)

The live tester makes a one-row temporary test dataframe from the typed English sentence, runs the selected translators, and displays their outputs in a table. For the Meta checker, it loads NLLB-200 distilled 600M and forces the Igbo output language token.

```python
# Live tester notebook
MODEL_LABELS = {
    'rnn_lstm_scratch': 'RNN LSTM from scratch',
    'transformer_scratch': 'Transformer from scratch',
    'opus_mt_pretrained': 'OPUS-MT pretrained',
    'opus_mt_fine_tuned': 'OPUS-MT fine-tuned',
    'nllb_checker': 'Meta NLLB-200 distilled checker',
}
```

```python
# Meta checker in the live tester
model_name = 'facebook/nllb-200-distilled-600M'
_NLLB_TOKENIZER = AutoTokenizer.from_pretrained(model_name, src_lang='eng_Latn')
_NLLB_MODEL = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
_NLLB_FORCED_BOS_TOKEN_ID = _NLLB_TOKENIZER.convert_tokens_to_ids('ibo_Latn')

generated = _NLLB_MODEL.generate(
    **encoded,
    forced_bos_token_id=_NLLB_FORCED_BOS_TOKEN_ID,
    max_new_tokens=100,
    num_beams=2,
)
```

The UI code creates checkboxes for the models, a text input, and a Translate button. This makes live comparison possible without setting up a Vercel app or a tunnel.

<div class="pagebreak"></div>

# 23. Limitations

The project has several limitations.

First, the data is not conversational. Bible text and JHW text are useful aligned corpora, yet my personal goal is learning Igbo for real communication. A future version should add conversational examples reviewed by fluent speakers.

Second, automatic metrics do not fully measure meaning. BLEU and chrF compare overlap with a reference. A good translation can use different wording. A bad translation can share many tokens with the reference and still shift meaning. Human evaluation by fluent Igbo speakers would make the evaluation stronger.

Third, the final comparison evaluates four project models. The live tester includes Meta NLLB as a checker, yet I did not run a full 6,390-row NLLB metric comparison in this final table. I kept the formal comparison focused on the four models I built or adapted.

Fourth, the final models are trained on limited compute. The fine-tuned OPUS-MT model used one epoch. Longer training, better hyperparameter search, and human-reviewed validation examples could improve quality.

Fifth, Igbo orthography and encoding issues matter. I kept the report figures visible in the PDF so the examples can be read without opening raw output tables.

<div class="pagebreak"></div>

# 24. Evidence Visible in This PDF

| Claim | Evidence shown inside this report |
|---|---|
| The final test set has 6,390 rows | Dataset composition table and the main metrics table both show 6,390 test rows |
| The fine-tuned OPUS model used all training rows | The fine-tuning settings table states 534,064 training rows and train sample size = all rows |
| All four project models were evaluated on the same rows | The prediction-count table shows 6,390 predictions for each model |
| Fine-tuned OPUS-MT was strongest | The main metric table and BLEU/chrF figure show the highest BLEU and chrF for fine-tuned OPUS-MT |
| The scratch Transformer beat direct OPUS-MT on BLEU | The hypothesis and results sections show Transformer BLEU 17.83 and direct OPUS-MT BLEU 15.26 |
| Direct OPUS-MT was still close on chrF | The main results table shows direct OPUS-MT chrF 36.13 and scratch Transformer chrF 36.02 |
| RNN under-translated | The length-ratio figure and table show RNN length ratio 0.861 |
| Bible and JHW performance differ | The source-specific table and figure separate Bible and JHW metrics |
| The live demo includes Meta NLLB | The live demo section shows the exact NLLB loading and generation code excerpt |

I include this table because the PDF should not depend on external inspection. Each major claim is supported by a table, figure, equation, or code excerpt that appears in the report.

<div class="pagebreak"></div>

# 25. Conclusion

The project answers my main research question. Transfer learning helped most when it was combined with fine-tuning on my cleaned English-Igbo corpus.

The final BLEU ranking was:

$$
\text{fine-tuned OPUS-MT} > \text{scratch Transformer} > \text{direct OPUS-MT} > \text{RNN LSTM}
$$

The final chrF ranking was similar, with direct OPUS-MT and the scratch Transformer nearly tied. This difference between BLEU and chrF is one of the most useful findings. It shows that model comparison depends on what kind of overlap the metric rewards.

My original expectation was too simple. I assumed direct pretraining would automatically beat my scratch Transformer. The results showed that a scratch model trained on the exact corpus can beat a direct pretrained model on BLEU. Fine-tuning then gave the pretrained model the advantage I expected.

For my personal goal of learning Igbo, the fine-tuned OPUS-MT model is the best model to keep using. The live Colab tester makes that practical because I can type sentences and compare outputs across the RNN, Transformer, OPUS models, and Meta NLLB.

## 25.1 AI Use Statement

I used OpenAI Codex as a debugging and development assistant throughout this project. Codex helped identify and fix notebook errors, troubleshoot training and runtime issues, clean up code structure, and improve reproducibility across the RNN, scratch Transformer, direct OPUS-MT, and fine-tuned OPUS-MT experiments. All model choices, experiment design, interpretation of results, and final report writing were ultimately decided and reviewed by me.

<div class="pagebreak"></div>

# 26. References

eBible.org. World English Bible copyright statement. https://ebible.org/eng-web/copyright.htm

Bahdanau, D., Cho, K., and Bengio, Y. 2015. Neural Machine Translation by Jointly Learning to Align and Translate. https://arxiv.org/abs/1409.0473

Hochreiter, S., and Schmidhuber, J. 1997. Long Short-Term Memory. Neural Computation, 9(8), 1735-1780. https://direct.mit.edu/neco/article/9/8/1735/6109/Long-Short-Term-Memory

Junczys-Dowmunt, M., Grundkiewicz, R., Dwojak, T., Hoang, H., Heafield, K., and others. 2018. Marian: Fast Neural Machine Translation in C++. https://arxiv.org/abs/1804.00344

NLLB Team. 2022. No Language Left Behind: Scaling Human-Centered Machine Translation. https://arxiv.org/abs/2207.04672

Papineni, K., Roukos, S., Ward, T., and Zhu, W. 2002. BLEU: a Method for Automatic Evaluation of Machine Translation. ACL Anthology. https://aclanthology.org/P02-1040/

Popovic, M. 2015. chrF: character n-gram F-score for automatic MT evaluation. ACL Anthology. https://aclanthology.org/W15-3049/

Tiedemann, J., and Thottingal, S. 2020. OPUS-MT: Building open translation services for the World. ACL Anthology. https://aclanthology.org/2020.eamt-1.61/

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., and Polosukhin, I. 2017. Attention Is All You Need. NeurIPS. https://papers.nips.cc/paper/7181-attention-is-all-you-need

Hugging Face model card. Helsinki-NLP/opus-mt-en-ig. https://huggingface.co/Helsinki-NLP/opus-mt-en-ig

Hugging Face model card. facebook/nllb-200-distilled-600M. https://huggingface.co/facebook/nllb-200-distilled-600M

Hugging Face Transformers documentation and library. https://huggingface.co/docs/transformers

SacreBLEU package used for BLEU and chrF computation. https://pypi.org/project/sacrebleu/

<div class="pagebreak"></div>

# 27. Appendix: Visible Code Excerpts

This appendix collects the most important implementation pieces in one place. These snippets are not meant to be every line of the project. They show the operations that matter for understanding the pipeline from the PDF alone.

## 28.1 Data Pairing and Cleaning

```python
# Align English and Igbo Bible verses by verseID.
english_df = load_english_verses().rename(columns={"verseText": "English"})
igbo_df = load_igbo_verses().rename(columns={"verseText": "Igbo"})

aligned_df = english_df.merge(
    igbo_df[["verseID", "Igbo"]],
    on="verseID",
    how="inner",
)

clean_bible_df = aligned_df[
    aligned_df["English"].str.strip().ne("") &
    aligned_df["Igbo"].str.strip().ne("")
].copy()
```

```python
# Remove blank English-Igbo sentence pairs from the larger JHW corpus.
clean_train = train_split.filter(
    lambda row: bool(
        row["English"] and row["English"].strip()
        and row["Igbo"] and row["Igbo"].strip()
    )
)
```

## 28.2 Architecture Summary Code

For a code-first architecture view, I can generate model summaries from the model objects instead of drawing boxes by hand. For the PyTorch RNN, the same idea can be written as a small shape table:

```python
def rnn_shape_summary(batch="B", source_len=40, target_len=40, embed_dim=256, hidden_dim=384):
    return [
        ("English token IDs", f"{batch} x {source_len}"),
        ("Source embedding", f"{batch} x {source_len} x {embed_dim}"),
        ("Bidirectional encoder output", f"{batch} x {source_len} x {hidden_dim * 2}"),
        ("Decoder hidden states", f"{batch} x {target_len} x {hidden_dim}"),
        ("Output logits", f"{batch} x {target_len} x target_vocab_size"),
    ]
```

For the Keras Transformer, `model.summary()` prints the layer names, output shapes, and parameter counts after the model is built:

```python
transformer = build_transformer_model(...)
transformer.summary(line_length=120)
```

For OPUS-MT, the Hugging Face config exposes the architecture directly:

```python
config = AutoConfig.from_pretrained("Helsinki-NLP/opus-mt-en-ig")
opus_architecture = {
    "encoder_layers": config.encoder_layers,
    "decoder_layers": config.decoder_layers,
    "d_model": config.d_model,
    "encoder_attention_heads": config.encoder_attention_heads,
    "decoder_attention_heads": config.decoder_attention_heads,
    "encoder_ffn_dim": config.encoder_ffn_dim,
    "decoder_ffn_dim": config.decoder_ffn_dim,
    "vocab_size": config.vocab_size,
}
```

## 28.3 RNN Encoder-Decoder Core

```python
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
            batch_first=True,
        )
        self.hidden_bridge = nn.Linear(hidden_dim * 2, hidden_dim)
        self.cell_bridge = nn.Linear(hidden_dim * 2, hidden_dim)
```

```python
for step in range(1, target_length):
    output, hidden, cell = self.decoder(input_token, hidden, cell)
    outputs[:, step] = output
    teacher_force = random.random() < teacher_forcing_ratio
    top1 = output.argmax(1)
    input_token = target[:, step] if teacher_force else top1
```

## 28.4 Transformer Masked Loss

```python
def masked_loss(y_true, y_pred):
    loss = keras.losses.sparse_categorical_crossentropy(
        y_true,
        y_pred,
        from_logits=True,
    )
    mask = tf.cast(tf.not_equal(y_true, 0), loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.maximum(tf.reduce_sum(mask), 1.0)
```

## 28.5 OPUS-MT Fine-Tuning Setup

```python
FINE_TUNE_TRAIN_SAMPLE_SIZE = None
FINE_TUNE_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 32
AUTO_FIND_BATCH_SIZE = True
USE_ADAFACTOR = True
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 96
GROUP_BY_LENGTH = True
```

```python
def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["English"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["Igbo"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
```

## 28.6 Shared Metric Computation

```python
def compute_metric_row(pred_df):
    refs = pred_df["Reference Igbo"].map(normalize_for_metric).tolist()
    preds = pred_df["Predicted Igbo"].fillna("").map(normalize_for_metric).tolist()

    exact_match = float(np.mean([pred == ref for pred, ref in zip(preds, refs)]))
    bleu = BLEU(tokenize="none", effective_order=True).corpus_score(preds, [refs]).score
    chrf = CHRF().corpus_score(preds, [refs]).score

    ref_lengths = np.array([max(token_count(text), 1) for text in pred_df["Reference Igbo"]])
    pred_lengths = np.array([token_count(text) for text in pred_df["Predicted Igbo"].fillna("")])
    length_ratio = float(np.mean(pred_lengths / ref_lengths))
```

<div class="pagebreak"></div>

# Appendix A: Full Data Cleaning Logic Summary

The Bible cleaning notebook does four essential operations:

1. Read English and Igbo SQL/VPL verse files.
2. Parse rows into verse IDs and verse text.
3. Inner join English and Igbo rows on `verseID`.
4. Remove rows with empty English or Igbo text.

The key alignment idea is:

$$
\text{BiblePairs} = \text{EnglishVerses} \bowtie_{verseID} \text{IgboVerses}
$$

The JHW cleaning notebook does three essential operations:

1. Load local JSONL files with `load_dataset("json")`.
2. Inspect split size, schema, samples, and blank values.
3. Filter rows where both `English` and `Igbo` are non-empty.

A reproducible combined dataset can be built from the cleaned files with this logic:

```python
from pathlib import Path
import pandas as pd

root = Path("Cleaned Data")
bible_train = pd.read_json(root / "Bible_train_cleaned.jsonl", lines=True)
bible_test = pd.read_json(root / "Bible_test_cleaned.jsonl", lines=True)
jhw_train = pd.read_json(root / "cleaned_JHW_train.jsonl", lines=True)
jhw_test = pd.read_json(root / "cleaned_JHW_test.jsonl", lines=True)

combined_train = pd.concat([bible_train, jhw_train], ignore_index=True)
combined_test = pd.concat([bible_test, jhw_test], ignore_index=True)

combined_train.to_json(root / "Combined_train.jsonl", orient="records", lines=True, force_ascii=False)
combined_test.to_json(root / "Combined_test.jsonl", orient="records", lines=True, force_ascii=False)
```

The final row counts in `dataset_composition.csv` verify the files used in model training and final evaluation.

<div class="pagebreak"></div>

# Appendix B: RNN Training and Inference Logic

The RNN training loop uses teacher forcing during training:

```python
for step in range(1, target_length):
    output, hidden, cell = self.decoder(input_token, hidden, cell)
    outputs[:, step] = output
    teacher_force = random.random() < teacher_forcing_ratio
    top1 = output.argmax(1)
    input_token = target[:, step] if teacher_force else top1
```

Final evaluation uses greedy decoding:

```python
for _ in range(max_decoding_steps):
    output, hidden, cell = model.decoder(input_token, hidden, cell)
    next_token = output.argmax(dim=1)
    input_token = next_token
```

This explains the RNN failure mode. During training, the decoder often sees the correct previous target token. During inference, it sees its own previous prediction. If it makes an early mistake, the remaining sentence can drift or repeat.

<div class="pagebreak"></div>

# Appendix C: Transformer Training Logic

The scratch Transformer uses shifted target sequences. The decoder input is the target prefix, and the decoder label is the next token.

Conceptually:

$$
y_{in} = (\langle bos \rangle, y_1, y_2, \ldots, y_{n-1})
$$

$$
y_{out} = (y_1, y_2, \ldots, y_n, \langle eos \rangle)
$$

The model learns:

$$
p(y_t \mid y_{<t}, x)
$$

The Keras notebook saves a best validation checkpoint and a final model artifact:

```python
callbacks = [
    keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
                                  restore_best_weights=True),
]

history = transformer.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
```

This matters because final evaluation loads the saved best model instead of retraining from scratch inside the report.

<div class="pagebreak"></div>

# Appendix D: OPUS-MT Fine-Tuning Logic

The fine-tuning notebook uses Hugging Face `Seq2SeqTrainer`. The preprocessing step tokenizes English inputs and Igbo targets separately:

```python
def preprocess_batch(batch):
    model_inputs = tokenizer(batch["English"], max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["Igbo"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
```

The trainer setup is:

```python
training_args = make_training_args(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=FINE_TUNE_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    predict_with_generate=False,
    fp16=USE_FP16,
    bf16=USE_BF16,
    optim="adafactor" if USE_ADAFACTOR else "adamw_torch",
    group_by_length=GROUP_BY_LENGTH,
    auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
    report_to="none",
)
```

The saved model info file records that the model trained on all 534,064 rows.

<div class="pagebreak"></div>

# Appendix E: Final Evaluation Reproducibility

The final evaluation is reproducible because all four models are treated the same way after training:

1. Build the same held-out test dataframe with 6,390 English-Igbo rows.
2. Generate one Igbo prediction for each English sentence from each model.
3. Normalize the reference and prediction text with the same regex tokenizer.
4. Compute exact match, BLEU, chrF, average prediction length, average reference length, and length ratio.
5. Save the same tables and figures used in this report.

The core evaluation logic is:

```python
prediction_frames = {
    "RNN": rnn_predictions,
    "Transformer": transformer_predictions,
    "OPUS pretrained": opus_pretrained_predictions,
    "OPUS fine-tuned": opus_fine_tuned_predictions,
}

for model_name, pred_df in prediction_frames.items():
    metrics.append(compute_metric_row(pred_df))
```

This is the part that matters for the report. The models differ in architecture and training, but the final measurement is shared.

<div class="pagebreak"></div>

# Appendix F: Future Potential Additions

Future work could add human evaluation. Fluent Igbo speakers could rate adequacy, fluency, and usefulness for language learning. A future extension could also add more conversational data, since my personal goal is speaking Igbo more naturally.

Possible future additions are:

| Improvement | Reason |
|---|---|
| Evaluate Meta NLLB on the full 6,390-row test set | Makes the fifth model part of the formal comparison |
| Fine-tune OPUS-MT for more than one epoch | Tests whether the best model still improves |
| Tune decoding settings | Beam search and length penalties may change BLEU and chrF |
| Add human-reviewed examples | Reduces risk from noisy automatic metrics |
| Build a small learner phrasebook set | Aligns the model with my personal goal |

These additions are outside the scope of this final submission, but they show how the project could grow from an automatic evaluation pipeline into a more learner-focused translation tool.